In [31]:
!pip install transformers sentencepiece --quiet


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
from transformers import pipeline

try:
    gen_bert = pipeline("text-generation", model="bert-base-uncased", device_map="auto")
    gen_roberta = pipeline("text-generation", model="roberta-base", device_map="auto")
    gen_bart = pipeline("text-generation", model="facebook/bart-base")
except ValueError:
    print("BERT and RoBERTa will fail as they are encoder-only models")

fill_bert = pipeline("fill-mask", model="bert-base-uncased")
fill_roberta = pipeline("fill-mask", model="roberta-base")
fill_bart = pipeline("fill-mask", model="facebook/bart-base")

qa_bert = pipeline("question-answering", model="bert-base-uncased")
qa_roberta = pipeline("question-answering", model="roberta-base")
qa_bart = pipeline("question-answering", model="facebook/bart-base")

BERT and RoBERTa will fail as they are encoder-only models


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Device set to use cpu
Device set to use cpu
Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able 

## Hypothesis - Experiment 1 (Text Generation)
BERT and RoBERTa will fail as they are encoder-only models because of which they do not have a decoder to generate tokens autoregressively. While, BART will succeed as it is an encoder-decoder model -- designed for generative tasks. 

In [33]:
prompt = "The future of Artificial Intelligence is"

try:
    models = {
            "BERT": gen_bert,
            "RoBERTa": gen_roberta,
            "BART": gen_bart
        }
except Exception:
    print("BERT and RoBERTa fail as they are encoder-only models.")

results_gen = {}

for name, model in models.items():
    print(f"\n--- {name} Text Generation ---")
    output = model(prompt, max_length=30)
    results_gen[name] = output
    print(output)

Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BERT and RoBERTa fail as they are encoder-only models.

--- BART Text Generation ---
[{'generated_text': 'The future of Artificial Intelligence is Remember Remember Rememberisha homosexuality Outs apprentices apprentices apprentices Wat filterawattsawatts homosexuality 2008LV Latinos Latinos additive additiveubipexubi slug LONG vill Vil Vil visuals warrants warrants warrants SensorBBCBeing699 Vil Vil VilRyanIV slug limitationslund slug699699 Sensorawatts slugBBCBBC Compet Compet homosexuality TormentBBC Wheeler OMG Compet Competawatts rational slug Competaicaic slugGoogle warrants unnoticedGoogleaicjust Jones turning Components Components slug Oxford Sensor Oxford Components slug Zub Oxford Sensor slug slugjust Components Compet Sensor Sensorake Components LONG Wheeler Vil Autumn Sensor Zub Sensor SensorGoogle slug Sensor ComponentsHill obliged Sensor699 Lav Components Jones Components Components Wheeler699 Compet Oxford Sensor Components Sensor Sensor Oxford Oxford Components Componen

## Hypothesis - Experiment 2 (Fill-Mask)
BERT and RoBERTa will perform well because of their pretraining task -- MLM. BART will work but would be less accurate as its objective is denoising, not pure MLM. 

In [36]:
masked_sentence = "The goal of Generative AI is to <mask> new content."

models_fill_1 = {
    "RoBERTa": fill_roberta,
    "BART": fill_bart
}

results_fill = {}

for name, model in models_fill_1.items():
    print(f"\n--- {name} Fill-Mask ---")
    try:
        output = model(masked_sentence)
        results_fill[name] = output
        print(output)
    except Exception as e:
        results_fill[name] = str(e)
        print("FAILED:", e)

masked_sentence_1 = "The goal of Generative AI is to [MASK] new content."
models_fill_2 = {
    "BERT": fill_bert
}
for name, model in models_fill_2.items():
    print(f"\n--- {name} Fill-Mask ---")
    try:
        output = model(masked_sentence_1)
        results_fill[name] = output
        print(output)
    except Exception as e:
        results_fill[name] = str(e)
        print("FAILED:", e)



--- RoBERTa Fill-Mask ---
[{'score': 0.37113118171691895, 'token': 5368, 'token_str': ' generate', 'sequence': 'The goal of Generative AI is to generate new content.'}, {'score': 0.3677138090133667, 'token': 1045, 'token_str': ' create', 'sequence': 'The goal of Generative AI is to create new content.'}, {'score': 0.08351466804742813, 'token': 8286, 'token_str': ' discover', 'sequence': 'The goal of Generative AI is to discover new content.'}, {'score': 0.02133519947528839, 'token': 465, 'token_str': ' find', 'sequence': 'The goal of Generative AI is to find new content.'}, {'score': 0.01652175933122635, 'token': 694, 'token_str': ' provide', 'sequence': 'The goal of Generative AI is to provide new content.'}]

--- BART Fill-Mask ---
[{'score': 0.07461544126272202, 'token': 1045, 'token_str': ' create', 'sequence': 'The goal of Generative AI is to create new content.'}, {'score': 0.06571853160858154, 'token': 244, 'token_str': ' help', 'sequence': 'The goal of Generative AI is to help

## Hypothesis - Experiment 3 (QA)
None of the models are fine-tuned for Question Answering(SQuAD), so BERT and RoBERTa will likely perform poorly.
BART may get closer because seq2seq models sometimes generalize better to extractive tasks.

In [37]:
context = "Generative AI poses significant risks such as hallucinations, bias, and deepfakes."
question = "What are the risks?"

models_qa = {
    "BERT": qa_bert,
    "RoBERTa": qa_roberta,
    "BART": qa_bart
}

results_qa = {}

for name, model in models_qa.items():
    print(f"\n--- {name} QA ---")
    try:
        output = model({"context": context, "question": question})
        results_qa[name] = output
        print(output)
    except Exception as e:
        results_qa[name] = str(e)
        print("FAILED:", e)



--- BERT QA ---
{'score': 0.00922791101038456, 'start': 62, 'end': 81, 'answer': 'bias, and deepfakes'}

--- RoBERTa QA ---


c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\pipelines\question_answering.py:395: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(


{'score': 0.008230676408857107, 'start': 72, 'end': 81, 'answer': 'deepfakes'}

--- BART QA ---
{'score': 0.02001120336353779, 'start': 0, 'end': 60, 'answer': 'Generative AI poses significant risks such as hallucinations'}


| Task | Model | Classification (Success/Failure) | Observation (What actually happened?) | Why did this happen? (Architectural Reason) |
|------|--------|----------------------------------|----------------------------------------|----------------------------------------------|
| **Generation** | **BERT** | Failure | Model fails to generate normal text; encoder-only model cannot perform causal generation. | BERT is an encoder-only bidirectional model, not trained for next-word generation. |
| | **RoBERTa** | Failure | Cannot generate text; no autoregressive head for generation. | RoBERTa is also encoder-only, optimized BERT, but still not a generative model. |
| | **BART** | Success | Generated a long continuation, although the text was incoherent and repetitive. | BART is an encoder–decoder model designed for sequence generation. |
| **Fill-Mask** | **BERT** | Success | Predicted “create”, “generate”, “produce”, “develop”, etc. | BERT was trained using Masked Language Modeling (MLM) with the `[MASK]` token. |
| | **RoBERTa** | Success | Predicted “generate”, “create”, “discover”, “find”, “provide”. | RoBERTa uses `<mask>` token and is heavily optimized for MLM, so predictions are strong. |
| | **BART** | Success | Predicted “create”, “help”, “provide”, “enable”, “improve”. | BART uses `<mask>` for denoising pretraining, so it can fill masks but less accurately. |
| **QA** | **BERT** | Partial Success / Weak | Returned only “deepfakes,” missing other risks. | Base BERT is not fine-tuned for QA tasks like SQuAD, so extraction is weak. |
| | **RoBERTa** | Failure | Did not return a meaningful or complete answer. | RoBERTa base model is also not QA-finetuned and cannot reliably extract answers. |
| | **BART** | Partial Success | Returned: “Generative AI poses significant risks such as hallucinations,” capturing part of the answer. | BART’s seq2seq architecture generalizes better but still lacks QA-specific finetuning. |
